# Constrained SLM Practicum — Google Colab Runner

Runs the whole project on Colab's free **T4 GPU**.

**Before you start:**
1. `Runtime ▸ Change runtime type ▸ Hardware accelerator: T4 GPU` → Save
2. Upload these to your **Google Drive** (in a folder called `slm_practicum`):
   - `slm_project_v2.zip` (the project code)
   - your data zips: `MOOC_1.zip`, `MOOC_2.zip`, …
3. Run the cells top to bottom.

⚠️ Colab wipes local files when the session ends — the last cell copies your results back to Drive, so **always run it before closing**.

In [ ]:
# 1) Verify the GPU is on
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU, then rerun.'

In [ ]:
# 2) Mount Google Drive (authorise when prompted)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/slm_practicum'   # <-- change if you used a different folder
import os
assert os.path.isdir(DRIVE_DIR), f'{DRIVE_DIR} not found - create it in Drive and upload your zips'
print('Found:', sorted(os.listdir(DRIVE_DIR)))

In [ ]:
# 3) Unpack the project code
import glob
!rm -rf /content/slm_project
proj_zip = sorted(glob.glob(f'{DRIVE_DIR}/slm_project*.zip'))[-1]
print('Using project zip:', proj_zip)
!unzip -q -o "{proj_zip}" -d /content/
%cd /content/slm_project
!ls

In [ ]:
# 4) Install dependencies
#    IMPORTANT: do NOT reinstall torch on Colab - it ships with the correct CUDA build.
#    We install everything else from requirements.txt minus the torch line.
!grep -vE '^torch' requirements.txt > /tmp/reqs_colab.txt
!pip install -q -r /tmp/reqs_colab.txt
!pip install -q peft datasets   # fine-tuning extras (Step 8)

# sanity check the heavy imports
import transformers, sentence_transformers, faiss, peft
print('transformers', transformers.__version__, '| peft', peft.__version__, '| all imports OK')

In [ ]:
# 5) Unzip the MOOC data into data/raw/
import shutil, glob
for z in sorted(glob.glob(f'{DRIVE_DIR}/MOOC*.zip')):
    print('copying', z)
    shutil.copy(z, 'data/raw/')
!python scripts/0_prepare_data.py

In [ ]:
# 6) Build the retrieval index (extract -> chunk -> FAISS + BM25)
#    First run downloads the ~80MB embedding model. A few minutes per module.
!python scripts/1_build_index.py

In [ ]:
# 7) Create the QA evaluation set
#    Writes a 3-question starter file and prints all available source names.
!python scripts/2_make_qa_set.py

### ✏️ Expand the QA set (important for real results)

The starter file has only 3 questions. For meaningful metrics you want **20–40** questions from your tutorial/exam material. Two options:

- **Option A:** edit `data/qa/qa.jsonl` here in Colab — double-click the file in the left Files panel.
- **Option B:** append items programmatically with the next cell (copy the pattern, keep the file in Drive so you never lose it).

Each line needs `question`, `reference` (gold answer), and `gold_sources` (exact source strings printed by the cell above).

In [ ]:
# 7b) OPTIONAL: append your own QA items (edit + rerun as needed)
import json

my_items = [
    # {
    #   'question': 'What is the difference between the Biba and Bell-LaPadula models?',
    #   'reference': 'Bell-LaPadula protects confidentiality (no read up, no write down) while Biba protects integrity (no read down, no write up).',
    #   'gold_sources': ['MOOC 1/Topic 4/M1.4.5 Biba Model - Article.docx',
    #                    'MOOC 1/Topic 4/M1.4.3 Bell-LaPadula Model - Article.docx'],
    # },
]

if my_items:
    with open('data/qa/qa.jsonl', 'a', encoding='utf-8') as f:
        for it in my_items:
            f.write(json.dumps(it, ensure_ascii=False) + '\n')
    print(f'appended {len(my_items)} items')

# if you keep a curated qa.jsonl in Drive, restore it instead:
# !cp "{DRIVE_DIR}/qa.jsonl" data/qa/qa.jsonl
print(sum(1 for _ in open('data/qa/qa.jsonl')), 'QA items total')

In [ ]:
# 8) RUN THE EXPERIMENTS (main result)
#    First run downloads the 3B generator (~2GB, a few minutes).
#    ~20-60s per question on the T4 depending on answer length.
!python scripts/3_run_experiments.py --tag qwen3b

In [ ]:
# 9) Inspect results
import pandas as pd
print(open('outputs/summary_qwen3b.txt').read())
df = pd.read_csv('outputs/results_per_question_qwen3b.csv')
df[df.stage == 'generation'][['question','method','grounding','hallucination_rate','latency_s','peak_vram_mb']].head(10)

### 🔬 Optional experiments

**RQ3 (model size):** edit `GEN_MODEL_NAME` in `src/config.py` (e.g. `Qwen/Qwen2.5-1.5B-Instruct` or `Qwen/Qwen2.5-7B-Instruct`), then rerun step 8 with a different `--tag` (e.g. `--tag qwen1.5b`). Compare the summary files.

**RQ2/RQ5 (LoRA fine-tuning):** run the next cell once your QA set has 20+ items.

In [ ]:
# 10) OPTIONAL: LoRA fine-tuning + evaluation of the fine-tuned condition
!python scripts/5_finetune_lora.py --epochs 3
!python scripts/3_run_experiments.py --adapter data/processed/lora_adapter --tag qwen3b_lora

In [ ]:
# 11) SAVE EVERYTHING BACK TO DRIVE - run this before closing the session!
import shutil, os
save_dir = f'{DRIVE_DIR}/results'
os.makedirs(save_dir, exist_ok=True)
for f in os.listdir('outputs'):
    if not f.startswith('.'):
        shutil.copy(f'outputs/{f}', save_dir)
shutil.copy('data/qa/qa.jsonl', f'{DRIVE_DIR}/qa.jsonl')   # keep your curated QA set safe
# keep the (small) LoRA adapter too, if trained
if os.path.isdir('data/processed/lora_adapter'):
    shutil.copytree('data/processed/lora_adapter', f'{save_dir}/lora_adapter', dirs_exist_ok=True)
print('Saved to Drive:', sorted(os.listdir(save_dir)))